In [2]:
# Recalculate metrics using your exact split method (60/20/20 with random_state=42)
# - Impute: categorical -> 'NA', numeric -> 0.0
# - Split: first 80/20, then 75/25 (=> 60/20/20 overall), random_state=42
# - Q1: single-feature ROC AUC (invert if AUC<0.5) on TRAIN only
# - Q2: DictVectorizer + LogisticRegression(liblinear, C=1.0, max_iter=1000) AUC on VAL
# - Q3: Precision/Recall vs threshold (0..1 step 0.01), intersection threshold
# - Q4: F1 vs threshold, best threshold

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score

# Load
df = pd.read_csv("course_lead_scoring.csv")

# Target
target = [c for c in df.columns if c.lower() == "converted"][0]
y_all = df[target].values
X_all = df.drop(columns=[target]).copy()

# Impute missing values
cat_cols = X_all.select_dtypes(include=["object"]).columns.tolist()
num_cols = X_all.select_dtypes(include=[np.number]).columns.tolist()
X_all[cat_cols] = X_all[cat_cols].fillna("NA")
X_all[num_cols] = X_all[num_cols].fillna(0.0)

# Split 60/20/20 with random_state=42 via two-stage split
df_train_full, df_test = train_test_split(
    pd.concat([X_all, pd.Series(y_all, name="__y__")], axis=1),
    test_size=0.2,
    random_state=42
)
df_train, df_val = train_test_split(
    df_train_full,
    test_size=0.25,
    random_state=42
)

# Separate back into X/y
def split_xy(frame):
    return frame.drop(columns=["__y__"]), frame["__y__"].values

X_train, y_train = split_xy(df_train)
X_val,   y_val   = split_xy(df_val)
X_test,  y_test  = split_xy(df_test)

# ---- Q1: ROC AUC per numeric feature (invert if <0.5) on TRAIN only
numeric_candidates = ["lead_score", "number_of_courses_viewed", "interaction_count", "annual_income"]
q1_aucs = {}
for col in numeric_candidates:
    s = X_train[col].astype(float).values
    if np.all(s == s[0]):
        s = s + 1e-9*np.random.default_rng(1).standard_normal(size=s.shape[0])
    auc = roc_auc_score(y_train, s)
    if auc < 0.5:
        s = -s
        auc = roc_auc_score(y_train, s)
    q1_aucs[col] = float(np.round(auc, 3))

q1_best = max(q1_aucs, key=q1_aucs.get)

# ---- Q2: DictVectorizer + Logistic Regression
dv = DictVectorizer(sparse=False)
Xtr_enc = dv.fit_transform(X_train.to_dict(orient="records"))
Xva_enc = dv.transform(X_val.to_dict(orient="records"))

lr = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000)
lr.fit(Xtr_enc, y_train)
val_proba = lr.predict_proba(Xva_enc)[:, 1]
auc_q2 = float(np.round(roc_auc_score(y_val, val_proba), 3))

# ---- Q3: Precision/Recall vs thresholds (0..1 step 0.01), intersection
thresholds = np.arange(0.0, 1.0 + 1e-12, 0.01)
precisions = []
recalls = []
for t in thresholds:
    preds = (val_proba >= t).astype(int)
    p = precision_score(y_val, preds, zero_division=1)
    r = recall_score(y_val, preds)
    precisions.append(p); recalls.append(r)

precisions = np.array(precisions); recalls = np.array(recalls)
t_intersect = float(np.round(thresholds[np.argmin(np.abs(precisions - recalls))], 3))

# ---- Q4: F1 vs thresholds
f1s = np.where((precisions+recalls)==0, 0.0, 2*precisions*recalls/(precisions+recalls))
t_best_f1 = float(np.round(thresholds[int(np.argmax(f1s))], 3))

{
    "Q1_auc_by_feature": q1_aucs,
    "Q1_best_feature": q1_best,
    "Q2_val_auc": auc_q2,
    "Q3_intersection_threshold": t_intersect,
    "Q4_best_f1_threshold": t_best_f1,
    "Split_sizes": {
        "train": int(len(X_train)),
        "val": int(len(X_val)),
        "test": int(len(X_test))
    }
}


{'Q1_auc_by_feature': {'lead_score': 0.63,
  'number_of_courses_viewed': 0.755,
  'interaction_count': 0.72,
  'annual_income': 0.529},
 'Q1_best_feature': 'number_of_courses_viewed',
 'Q2_val_auc': 0.855,
 'Q3_intersection_threshold': 0.64,
 'Q4_best_f1_threshold': 0.57,
 'Split_sizes': {'train': 876, 'val': 293, 'test': 293}}

In [3]:
# Continue with Question 5 and 6 using the same split and data prep

from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

# Combine train + val for CV (full_train)
X_full_train = pd.concat([X_train, X_val], axis=0)
y_full_train = np.concatenate([y_train, y_val])

# ---- Q5: 5-Fold CV with C=1.0
kf = KFold(n_splits=5, shuffle=True, random_state=1)
cv_scores = []

for train_idx, val_idx in kf.split(X_full_train):
    X_tr, X_va = X_full_train.iloc[train_idx], X_full_train.iloc[val_idx]
    y_tr, y_va = y_full_train[train_idx], y_full_train[val_idx]
    
    dv_fold = DictVectorizer(sparse=False)
    X_tr_enc = dv_fold.fit_transform(X_tr.to_dict(orient="records"))
    X_va_enc = dv_fold.transform(X_va.to_dict(orient="records"))
    
    model = LogisticRegression(solver="liblinear", C=1.0, max_iter=1000)
    model.fit(X_tr_enc, y_tr)
    y_va_pred = model.predict_proba(X_va_enc)[:, 1]
    
    auc = roc_auc_score(y_va, y_va_pred)
    cv_scores.append(auc)

std_q5 = float(np.round(np.std(cv_scores), 3))

# ---- Q6: Hyperparameter tuning (C in [0.000001, 0.001, 1])
C_values = [0.000001, 0.001, 1]
results = {}

for C in C_values:
    scores = []
    for train_idx, val_idx in kf.split(X_full_train):
        X_tr, X_va = X_full_train.iloc[train_idx], X_full_train.iloc[val_idx]
        y_tr, y_va = y_full_train[train_idx], y_full_train[val_idx]

        dv_fold = DictVectorizer(sparse=False)
        X_tr_enc = dv_fold.fit_transform(X_tr.to_dict(orient="records"))
        X_va_enc = dv_fold.transform(X_va.to_dict(orient="records"))

        model = LogisticRegression(solver="liblinear", C=C, max_iter=1000)
        model.fit(X_tr_enc, y_tr)
        y_va_pred = model.predict_proba(X_va_enc)[:, 1]
        scores.append(roc_auc_score(y_va, y_va_pred))
    
    mean_score = np.round(np.mean(scores), 3)
    std_score = np.round(np.std(scores), 3)
    results[C] = (mean_score, std_score)

best_C = max(results.keys(), key=lambda c: (results[c][0], -results[c][1], -c))

{
    "Q5_cv_std": std_q5,
    "Q6_results": results,
    "Q6_best_C": best_C
}


{'Q5_cv_std': 0.008,
 'Q6_results': {1e-06: (0.542, 0.042),
  0.001: (0.87, 0.013),
  1: (0.827, 0.008)},
 'Q6_best_C': 0.001}